# Inside Steam - Statistical Analysis

This notebook uses statistical methods to test relationships and differences identified during the exploratory and SQL analyses.

The objective is to determine whether observed differences in market reach, player engagement and satisfaction across Steam market segments are statistically meaningful.

Because several project variables are highly skewed, bounded or based on unevenly sized groups, non-parametric statistical methods are used when appropriate.

In [1]:
import pandas as pd
import numpy as np

from scipy import stats

In [2]:
file_path = "../data/processed/games_cleaned_march2025.csv"

df = pd.read_csv(
    file_path,
    low_memory=False
)

df.shape

(94948, 48)

## 1. Player Satisfaction Across Price Bands

### Research question

Does player satisfaction differ significantly across Steam price segments?

### Hypotheses

**H0 (null hypothesis):**  
The distribution of player satisfaction is the same across price bands.

**H1 (alternative hypothesis):**  
At least one price band has a different player satisfaction distribution.

Because player review scores are bounded and may not follow a normal distribution, the Kruskal-Wallis test is used instead of a standard one-way ANOVA.

To improve the reliability of the comparison, only games with at least 50 total reviews are included.

In [3]:
reception_price = df[
    (df["pct_pos_total"].notna()) &
    (df["num_reviews_total"] >= 50)
][["price_band", "pct_pos_total"]].copy()

In [4]:
price_reception_summary = (
    reception_price
    .groupby("price_band", observed=True)["pct_pos_total"]
    .agg(
        games="count",
        median="median",
        mean="mean"
    )
    .round(2)
)

price_reception_summary

,games,median,mean
price_band,,,
$0.01-$4.99,7240,82.0,78.30
$10-$19.99,6402,84.0,80.49
$20-$39.99,2181,82.0,79.58
$40+,464,80.0,76.76
$5-$9.99,5543,83.0,79.68
Free,7411,82.0,78.18


In [5]:
price_order = [
    "Free",
    "$0.01-$4.99",
    "$5-$9.99",
    "$10-$19.99",
    "$20-$39.99",
    "$40+"
]

groups = [
    reception_price.loc[
        reception_price["price_band"] == band,
        "pct_pos_total"
    ]
    for band in price_order
]

h_stat, p_value = stats.kruskal(*groups)

print("Kruskal-Wallis H-statistic:", h_stat)
print("p-value:", p_value)

Kruskal-Wallis H-statistic: 106.8609360073908
p-value: 1.8863580255037513e-21


In [6]:
n = len(reception_price)
k = len(groups)

epsilon_squared = (h_stat - k + 1) / (n - k)

print("Sample size:", n)
print("Epsilon-squared:", epsilon_squared)

Sample size: 29241
Epsilon-squared: 0.0034842119379986593


### Interpretation

The Kruskal-Wallis test showed a statistically significant difference in player satisfaction across price bands:

- **H-statistic:** 106.86
- **p-value:** 1.89 × 10⁻²¹
- **Epsilon-squared:** 0.0035

The very small p-value indicates that at least one price band has a different satisfaction distribution.

However, the effect size is extremely small. Median positive review scores range only from 80% to 84%, and epsilon-squared is approximately 0.0035.

This suggests that although price segments differ statistically, price band has very limited practical importance for explaining player satisfaction.

The result illustrates the difference between **statistical significance** and **practical significance**, especially with a large sample size.

## 2. Market Reach and Player Satisfaction

### Research question

Is estimated market reach associated with player satisfaction?

### Hypotheses

**H0 (null hypothesis):**  
There is no monotonic relationship between estimated market reach and player satisfaction.

**H1 (alternative hypothesis):**  
There is a monotonic relationship between estimated market reach and player satisfaction.

Spearman's rank correlation is used because estimated owner counts are highly skewed and derived from broad ownership ranges rather than exact values.

Only games with usable owner estimates and at least 50 total reviews are included.

In [7]:
reach_reception = df[
    (df["estimated_owners_midpoint"].notna()) &
    (df["pct_pos_total"].notna()) &
    (df["num_reviews_total"] >= 50)
][
    ["estimated_owners_midpoint", "pct_pos_total"]
].copy()

print("Sample size:", len(reach_reception))

reach_reception.describe().round(2)

Sample size: 24946


,estimated_owners_midpoint,pct_pos_total
count,2.494600e+04,24946.00
mean,3.241971e+05,78.86
std,3.159479e+06,14.71
min,1.000000e+04,1.00
25%,1.000000e+04,71.00
50%,3.500000e+04,82.00
75%,1.500000e+05,90.00
max,3.500000e+08,100.00


In [8]:
spearman_rho, p_value = stats.spearmanr(
    reach_reception["estimated_owners_midpoint"],
    reach_reception["pct_pos_total"]
)

print("Spearman correlation:", spearman_rho)
print("p-value:", p_value)

Spearman correlation: 0.008544913214780465
p-value: 0.1771550849191031


### Interpretation

Spearman's rank correlation showed almost no relationship between estimated market reach and player satisfaction:

- **Spearman correlation:** 0.0085
- **p-value:** 0.177

The correlation coefficient is extremely close to zero, indicating that games with higher estimated ownership do not consistently receive higher or lower player satisfaction scores.

Because the p-value is greater than 0.05, the null hypothesis is not rejected.

This suggests that, within this sample, estimated market reach and player satisfaction behave as largely independent dimensions of game performance.

The result supports the broader project finding that high audience reach does not necessarily imply strong player reception.

## 3. Player Engagement and Satisfaction

### Research question

Is player engagement associated with player satisfaction?

### Hypotheses

**H0 (null hypothesis):**  
There is no monotonic relationship between lifetime playtime and player satisfaction.

**H1 (alternative hypothesis):**  
There is a monotonic relationship between lifetime playtime and player satisfaction.

Spearman's rank correlation is used because lifetime playtime is highly skewed and contains extreme values.

Only games with usable lifetime playtime data, non-extreme playtime values and at least 50 total reviews are included.

In [9]:
engagement_reception = df[
    (df["has_lifetime_playtime"] == True) &
    (df["extreme_playtime"] == False) &
    (df["pct_pos_total"].notna()) &
    (df["num_reviews_total"] >= 50)
][
    ["median_playtime_forever", "pct_pos_total"]
].copy()

engagement_reception["median_playtime_hours"] = (
    engagement_reception["median_playtime_forever"] / 60
)

print("Sample size:", len(engagement_reception))

engagement_reception[
    ["median_playtime_hours", "pct_pos_total"]
].describe().round(2)

Sample size: 7091


,median_playtime_hours,pct_pos_total
count,7091.00,7091.00
mean,7.54,79.93
std,12.49,13.78
min,0.02,13.00
25%,1.18,73.00
50%,3.67,83.00
75%,7.79,91.00
max,108.67,99.00


In [10]:
spearman_rho, p_value = stats.spearmanr(
    engagement_reception["median_playtime_hours"],
    engagement_reception["pct_pos_total"]
)

print("Spearman correlation:", spearman_rho)
print("p-value:", p_value)

Spearman correlation: 0.09929164207373463
p-value: 5.274845966555135e-17


### Interpretation

Spearman's rank correlation identified a statistically significant but very weak positive relationship between player engagement and satisfaction:

- **Spearman correlation:** 0.0993
- **p-value:** 5.27 × 10⁻¹⁷

The very small p-value indicates that the relationship is statistically detectable.

However, the correlation coefficient is close to zero, showing that the association is weak in practical terms.

Games with higher lifetime playtime therefore tend to have slightly higher player satisfaction, but engagement alone explains very little about differences in review scores.

Compared with the previous analysis, engagement shows a slightly stronger relationship with satisfaction than estimated market reach, although both remain largely distinct dimensions of performance.

Results are based only on games with usable, non-extreme lifetime playtime data and at least 50 total reviews.

## 4. Market Reach Across Price Bands

### Research question

Does estimated market reach differ significantly across Steam price segments?

### Hypotheses

**H0 (null hypothesis):**  
The distribution of estimated market reach is the same across price bands.

**H1 (alternative hypothesis):**  
At least one price band has a different estimated market reach distribution.

Because estimated ownership is highly skewed and derived from broad ownership ranges, the Kruskal-Wallis test is used to compare price segments without assuming normal distributions.

In [11]:
reach_price = df[
    df["estimated_owners_midpoint"].notna()
][
    [
        "price_band",
        "estimated_owners_midpoint",
        "owners_lower"
    ]
].copy()

In [13]:
reach_price_summary = (
    reach_price
    .groupby("price_band", observed=True)
    .agg(
        games=("estimated_owners_midpoint", "count"),
        median_owner_midpoint=("estimated_owners_midpoint", "median"),
        games_100k_plus=("owners_lower", lambda x: (x >= 100000).sum())
    )
)

reach_price_summary["share_100k_plus_pct"] = (
    reach_price_summary["games_100k_plus"]
    / reach_price_summary["games"]
    * 100
).round(2)

reach_price_summaryreach_price_summary = (
    reach_price
    .groupby("price_band", observed=True)
    .agg(
        games=("estimated_owners_midpoint", "count"),
        median_owner_midpoint=("estimated_owners_midpoint", "median"),
        games_100k_plus=("owners_lower", lambda x: (x >= 100000).sum())
    )
)

reach_price_summary["share_100k_plus_pct"] = (
    reach_price_summary["games_100k_plus"]
    / reach_price_summary["games"]
    * 100
).round(2)

reach_price_summary

,games,median_owner_midpoint,games_100k_plus,share_100k_plus_pct
price_band,,,,
$0.01-$4.99,38704,10000.0,1807,4.67
$10-$19.99,12683,10000.0,1944,15.33
$20-$39.99,3320,10000.0,990,29.82
$40+,861,10000.0,249,28.92
$5-$9.99,18769,10000.0,1289,6.87
Free,6955,10000.0,1384,19.90


Because ownership estimates are provided as broad ranges, the median owner midpoint is identical across price bands and is not sufficiently informative for this comparison.

The analysis therefore focuses on a more interpretable reach threshold: whether a game has at least 100,000 estimated owners.

A Chi-square test of independence is used to test whether high market reach is associated with price segment.

In [14]:
reach_price["high_reach_100k"] = (
    reach_price["owners_lower"] >= 100000
)

In [15]:
contingency_table = pd.crosstab(
    reach_price["price_band"],
    reach_price["high_reach_100k"]
)

contingency_table

high_reach_100k,False,True
price_band,,
$0.01-$4.99,36897,1807
$10-$19.99,10739,1944
$20-$39.99,2330,990
$40+,612,249
$5-$9.99,17480,1289
Free,5571,1384


In [16]:
chi2, p_value, dof, expected = stats.chi2_contingency(
    contingency_table
)

print("Chi-square statistic:", chi2)
print("Degrees of freedom:", dof)
print("p-value:", p_value)

Chi-square statistic: 4581.124210469202
Degrees of freedom: 5
p-value: 0.0


In [17]:
n = contingency_table.to_numpy().sum()

rows, cols = contingency_table.shape

cramers_v = np.sqrt(
    chi2 / (n * min(rows - 1, cols - 1))
)

print("Sample size:", n)
print("Cramer's V:", cramers_v)

Sample size: 81292
Cramer's V: 0.23738984088518572


### Interpretation

The Chi-square test showed a highly significant association between price band and the probability of reaching at least 100,000 estimated owners:

- **Chi-square statistic:** 4581.12
- **Degrees of freedom:** 5
- **p-value:** < 0.001
- **Cramer's V:** 0.237

The association is statistically significant and meaningfully stronger than the effect observed between price band and player satisfaction.

Among paid games, the share reaching 100k+ estimated owners increases substantially across higher price bands, from 4.67% in the $0.01-$4.99 group to around 29% in the $20-$39.99 and $40+ groups.

Free-to-play games also show relatively strong reach, with 19.90% reaching the 100k+ threshold.

This suggests that price segment and market reach are associated, although the relationship should not be interpreted as causal.